after linear regression comes logistic regression

In [1]:
import math
import random

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def binary_cross_entropy(actual,predicted):
    value = -(actual * math.log(min(max(predicted, 1e-15), 1 - 1e-15)) +
        (1 - actual) * math.log(1 - min(max(predicted, 1e-15), 1 - 1e-15)))
    return value

class Neuron:
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias

def generate_pass_fail(n=1000):
    data = []
    for _ in range(n):
        hours = random.uniform(0, 10)
        sleep = random.uniform(4, 9)
        z = 0.8*hours + 0.5*sleep - 7
        p = sigmoid(z)
        y = random.random() < p
        data.append(((hours, sleep), int(y)))
    return data

def forward_pass(x, neuron):
    z = 0
    for xi, wi in zip(x, neuron.weights):
        z += xi * wi
    z += neuron.bias

    y_hat = sigmoid(z)
    return y_hat

## Gradient of the loss

With binary cross-entropy loss and a sigmoid output, the chain rule gives:

$$
\frac{\partial L}{\partial \hat{y}} = \frac{\hat{y} - y}{\hat{y}(1-\hat{y})}
$$

$$
\frac{\partial \hat{y}}{\partial z} = \hat{y}(1-\hat{y})
$$

Multiplying them, the $\hat{y}(1-\hat{y})$ terms cancel:

$$
\frac{\partial L}{\partial z} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} = \hat{y} - y
$$

So the gradient at $z$ is just the prediction error. Since $z = \sum_i w_i x_i + b$, the weight and bias gradients follow directly:

$$
\frac{\partial L}{\partial w_i} = (\hat{y} - y)\, x_i \qquad \frac{\partial L}{\partial b} = \hat{y} - y
$$

which is why each update is `learning_rate * error * x[i]` for the weights and `learning_rate * error` for the bias.

In [2]:

data = generate_pass_fail(5)

neuron = Neuron(weights=[0.1, -0.2], bias=0.0)

for x, y in data:
    y_hat = forward_pass(x, neuron)
    loss = binary_cross_entropy(y, y_hat)
    print(f"x={x}, y={y}, ŷ={y_hat:.3f}, loss={loss:.3f}")


x=(3.7467462903108717, 6.465544938194695), y=1, ŷ=0.285, loss=1.254
x=(8.480758650019629, 4.4222793002848775), y=1, ŷ=0.491, loss=0.712
x=(0.45554248522913343, 8.66370752298037), y=0, ŷ=0.156, loss=0.170
x=(7.457919249851383, 7.747227443706338), y=1, ŷ=0.309, loss=1.174
x=(9.626581698611977, 7.2006105332807095), y=1, ŷ=0.383, loss=0.960


In [3]:
def train(neuron, data, lr=0.01, epochs=100):
    for epoch in range(epochs):
        total_loss = 0
        for x, y in data:
            # forward
            y_hat = forward_pass(x, neuron)
            
            # compute loss
            loss = binary_cross_entropy(y, y_hat)
            total_loss += loss
            
            # backprop
            error = y_hat - y
            for i in range(len(neuron.weights)):
                neuron.weights[i] -= lr * error * x[i]
            neuron.bias -= lr * error
        
        avg_loss = total_loss / len(data)
        if epoch % 50 == 0:
            print(f"Epoch {epoch}: avg loss = {avg_loss:.4f}")

In [4]:
data = generate_pass_fail(500)
neuron = Neuron(weights=[0.0, 0.0], bias=0.0)

train(neuron, data, lr=0.01, epochs=500)


Epoch 0: avg loss = 0.5367
Epoch 50: avg loss = 0.4138
Epoch 100: avg loss = 0.4051
Epoch 150: avg loss = 0.4039
Epoch 200: avg loss = 0.4037
Epoch 250: avg loss = 0.4037
Epoch 300: avg loss = 0.4037
Epoch 350: avg loss = 0.4037
Epoch 400: avg loss = 0.4037
Epoch 450: avg loss = 0.4037


In [5]:
# make predictions after training
for x, y in data[:5]:
    y_hat = forward_pass(x, neuron)
    print(f"x={x}, y={y}, ŷ={y_hat:.3f}")


x=(4.192401601168648, 7.100066650756446), y=1, ŷ=0.511
x=(5.875732793919063, 4.506114032880066), y=0, ŷ=0.530
x=(6.309724557624387, 5.967693181527039), y=1, ŷ=0.777
x=(3.32658675604262, 8.164886506498048), y=0, ŷ=0.467
x=(1.65850391428648, 7.69453551420491), y=0, ŷ=0.143


## Why log odds and BCE

We assume a linear relationship between the inputs and the log odds:

$$
\log \frac{p}{1-p} = \sum_i w_i x_i + b = z
$$

Using log odds makes the relationship additive. On the odds themselves, each feature's effect would be multiplicative:

$$
\frac{p}{1-p} = e^{z} = e^{b} \prod_i e^{w_i x_i}
$$

Solving for $p$ gives the sigmoid, $p = \frac{1}{1 + e^{-z}}$.

Binary cross-entropy is just the negative log-likelihood of the data under this model:

$$
L = -\sum_i \Big[ y_i \log \hat{y}_i + (1 - y_i) \log (1 - \hat{y}_i) \Big]
$$

so minimising BCE is the same as maximising the likelihood.

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

X = torch.tensor([[1., 2.],
                  [2., 1.],
                  [2., 3.],
                  [3., 2.]])
y = torch.tensor([[0.], [0.], [1.], [1.]])

model = nn.Linear(2, 1)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for _ in range(1000):
    optimizer.zero_grad()

    logits = model(X)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

print(model.weight, model.bias)


Parameter containing:
tensor([[1.6529, 1.6529]], requires_grad=True) Parameter containing:
tensor([-6.3417], requires_grad=True)
